<a href="https://colab.research.google.com/github/hibbo67/ai-engineering-101/blob/main/day21_first_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb google-genai
from google.colab import userdata
from google import genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

# 1. Load keys
api_keys = [
    userdata.get("GEMINI_KEY_1"),
    userdata.get("GEMINI_KEY_2"),
    userdata.get("GEMINI_KEY_3"),
]
# Filter out any None values (if a secret is missing)
ALL_KEYS = [k for k in api_keys if k]

# 2. Client for embeddings (uses the first key)
embed_client = genai.Client(api_key=ALL_KEYS[0])

# 3. Embedding function
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        pass # Silences the Chroma deprecation warning

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            result = embed_client.models.embed_content(
                model="gemini-embedding-2",
                contents=[text],
            )
            vectors.append([float(x) for x in result.embeddings[0].values])
        return vectors

print(f"Setup complete. {len(ALL_KEYS)} keys loaded for manual rotation.")

Setup complete. 3 keys loaded for manual rotation.


In [ ]:
docs = [
    "Our refund policy allows returns within 30 days of purchase.",
    "Shipping is free for orders over $50.",
    "We ship to the US, Canada, and EU only.",
    "Customer support is available 9am–5pm EST, Monday to Friday.",
    "Digital products cannot be refunded once downloaded.",
    "To initiate a return, please contact support with your order number.",
]

In [ ]:
# Get the collection (creates it if it doesn't exist, avoids crash if it does)
col = db.get_or_create_collection(
    name="support_kb",
    embedding_function=GeminiEmbeddingFunction()
)

# Add documents (assuming 'docs' is still defined in memory from earlier)
col.add(
    ids=[f"doc{i}" for i in range(len(docs))],
    documents=docs,
)

print(f"Stored {col.count()} documents.")

/tmp/ipykernel_849/2968062321.py:4: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embedding_function=GeminiEmbeddingFunction()


Stored 6 documents.


In [ ]:
import time

def answer(question):
    # --- PHASE 1: RETRIEVE ---
    hits = col.query(query_texts=[question], n_results=2)
    retrieved_docs = hits["documents"][0]
    distances = hits["distances"][0]
    top_distance = distances[0]

    # --- PHASE 2: GUARDRAIL ---
    if top_distance > 0.75:
        return "I don't know — that information isn't in my documents.", None

    # --- PHASE 3: AUGMENT ---
    context_parts = [f"[{i+1}] {doc}" for i, doc in enumerate(retrieved_docs)]
    context = "\n\n".join(context_parts)

    prompt = f"""Answer the user's question using ONLY the context below.
If the context does not contain the answer, you must say "I don't know."
Cite your sources using the bracketed numbers [1], [2], etc.

Context:
{context}

Question: {question}
"""

    # --- PHASE 4: GENERATE (Modern SDK + Manual Rotation) ---
    last_error = None

    # Try each key until one works
    for key in ALL_KEYS:
        try:
            # Create a temporary client for this specific key
            temp_client = genai.Client(api_key=key)

            response = temp_client.models.generate_content(
                model="gemini-3.6-flash",
                config={"thinking_config": {"thinking_level": "low"}},
                contents=prompt
            )

            # If successful, return immediately
            return response.text, context

        except Exception as e:
            # If it's a quota error (429) or invalid key (400), try the next key
            if "429" in str(e) or "400" in str(e) or "API_KEY_INVALID" in str(e):
                print(f"  [Key failed, rotating...]")
                last_error = e
                time.sleep(1) # Brief pause before trying the next key
            else:
                # If it's a different error (like a safety block), raise it immediately
                raise e

    # If all keys failed
    raise Exception(f"All API keys failed. Last error: {last_error}")

In [ ]:
questions = [
    "What's your refund policy?",        # In the docs
    "Do you ship to Australia?",         # Not in the docs (we only ship to US/CA/EU)
    "What's your favorite color?"        # Completely irrelevant (should trigger guardrail)
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    ans, ctx = answer(q)
    print(f"A: {ans}")


Q: What's your refund policy?
A: Our refund policy allows returns within 30 days of purchase [1]. However, digital products cannot be refunded once they have been downloaded [2].

Q: Do you ship to Australia?
A: No, we ship to the US, Canada, and EU only [1].

Q: What's your favorite color?
A: I don't know — that information isn't in my documents.
